# 🌾 Atelier IA Agricole — 04. OV: détection ouverte, segmentation et tracking

**OV** (*Open Vocabulary*) désigne des modèles de détection qui cherchent un objet à partir
d'un **mot** plutôt que d'une liste figée de catégories. Ce notebook montre trois briques
visuelles utiles au champ:
- **bounding box**: repérer un objet avec un mot-clé (« a leaf », « fruit »…) ;
- **segmentation**: isoler l'objet dans l'image ;
- **tracking**: suivre cet objet sur une petite séquence d'images.

Le modèle utilisé (**OWL-ViT base**, ~150M paramètres) est volontairement **petit et rapide**,
pour tourner confortablement sur le CPU gratuit de Google Colab. Le code reste minimal: le but
est d'apprendre à **utiliser** le modèle et à observer l'effet du **prompt** (le mot-clé donné
au détecteur), pas d'écrire beaucoup de code.


In [ ]:
# === Configuration de l'environnement (exécuter en premier) ===
import os, sys, subprocess

# Sommes-nous dans Google Colab?
try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except Exception:
    IS_COLAB = False

# MODE_DEMO = True  -> utilise de tout petits modèles / jeux de données réduits
# (utile pour tester rapidement, ou hors GPU). Mettez False pour l'atelier complet.
# La variable d'environnement ATELIER_DEMO permet de forcer le mode pour les tests.
MODE_DEMO = os.environ.get("ATELIER_DEMO", "0") == "1"

def pip_install(*paquets):
    """Installe des paquets pip de façon silencieuse (Colab ou local)."""
    cmd = [sys.executable, "-m", "pip", "install", "-q", *paquets]
    print("Installation:", " ".join(paquets), "...")
    subprocess.run(cmd, check=False)

print(f"IS_COLAB = {IS_COLAB}")
print(f"MODE_DEMO = {MODE_DEMO}")
print("Python:", sys.version.split()[0])


In [ ]:
pip_install("transformers>=4.56", "accelerate", "pillow", "opencv-contrib-python")
import numpy as np
import cv2
from PIL import ImageDraw
print("✅ opencv", cv2.__version__)


In [ ]:
pip_install("kagglehub", "pandas")
import kagglehub

def telecharger_dataset_kaggle(reference):
    """Télécharge un jeu de données Kaggle public et renvoie son dossier local.
    Renvoie None si Kaggle est injoignable (un échantillon de secours prend alors le relais)."""
    try:
        dossier = kagglehub.dataset_download(reference)
        print(f"✅ Jeu de données Kaggle prêt: {reference}")
        return dossier
    except Exception as e:
        print(f"⚠️ Kaggle indisponible ({e}) → utilisation d'un échantillon de secours.")
        return None


In [ ]:
pip_install("pillow")
import os, random, io, urllib.request
from PIL import Image

REFERENCE_IMAGES_KAGGLE = "rashikrahmanpritom/plant-disease-recognition-dataset"
URLS_SECOURS = [
    "https://upload.wikimedia.org/wikipedia/commons/thumb/1/16/Leaf_curl_on_peach.jpg/960px-Leaf_curl_on_peach.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/thumb/f/f6/Jowar_leaf_infested_by_pest_or_disease.jpg/960px-Jowar_leaf_infested_by_pest_or_disease.jpg",
]

def echantillon_images_plantes(n=10):
    """Renvoie une liste de (image PIL, label): n photos du jeu Kaggle Healthy/Powdery/Rust ;
    à défaut, quelques URLs publiques ; et en dernier recours une image synthétique hors-ligne.
    Ainsi la liste renvoyée n'est jamais vide, même sans réseau."""
    dossier = telecharger_dataset_kaggle(REFERENCE_IMAGES_KAGGLE)
    if dossier:
        try:
            base = os.path.join(dossier, "Test", "Test")
            fichiers = []
            for classe in sorted(os.listdir(base)):
                dossier_c = os.path.join(base, classe)
                if not os.path.isdir(dossier_c):
                    continue
                for nom in sorted(os.listdir(dossier_c))[:20]:
                    fichiers.append((os.path.join(dossier_c, nom), classe))
            random.Random(42).shuffle(fichiers)
            images = [(Image.open(chemin).convert("RGB"), label) for chemin, label in fichiers[:n]]
            if images:
                return images
            print("⚠️ Jeu Kaggle vide/inattendu → images de secours.")
        except Exception as e:
            print(f"⚠️ Lecture du jeu Kaggle impossible ({e}) → images de secours.")

    entetes = {"User-Agent": "Mozilla/5.0 (AtelierIA-Agricole; educatif)"}
    images = []
    for url in URLS_SECOURS[:n]:
        try:
            req = urllib.request.Request(url, headers=entetes)
            with urllib.request.urlopen(req, timeout=20) as r:
                images.append((Image.open(io.BytesIO(r.read())).convert("RGB"), "inconnu"))
        except Exception as e:
            print("⚠️ Image ignorée:", e)

    # Dernier recours 100% hors-ligne: si Kaggle ET les URLs échouent, on fabrique au moins
    # une image synthétique pour que le notebook aille jusqu'au bout (pas d'IndexError sur
    # echantillon[0], liste jamais vide).
    if not images:
        from PIL import ImageDraw
        print("⚠️ Aucune image en ligne → image de secours générée localement.")
        for _ in range(max(1, min(n, 3))):
            img = Image.new("RGB", (256, 256), (150, 170, 90))
            ImageDraw.Draw(img).ellipse([40, 40, 216, 216], fill=(70, 110, 50))
            images.append((img, "inconnu"))
    return images


## 1. Charger des images agricoles (Kaggle)

On réutilise le jeu **Plant Disease Recognition** (Kaggle, `Healthy` / `Powdery` / `Rust`) déjà
vu au notebook 02, et on garde la première photo pour la démonstration pas à pas.


In [ ]:
N_EXEMPLES = 3 if MODE_DEMO else 10
echantillon = echantillon_images_plantes(N_EXEMPLES)
image, vrai_label = echantillon[0]
print(f"{len(echantillon)} images chargées. Photo de démonstration: {vrai_label}")
image.thumbnail((480, 480))
image


## 2. Trouver un objet avec du texte (détection ouverte)

On donne un mot, et le modèle cherche dans l'image l'objet correspondant — sans avoir été
entraîné spécifiquement sur ce mot. On garde toujours la **meilleure** boîte trouvée, même si
sa confiance est faible (utile sur une photo dense, où « leaf » désigne une zone plutôt qu'un
objet unique): on affiche alors un avertissement plutôt que de planter.


In [ ]:
from transformers import pipeline

MODELE_DETECTION = "google/owlvit-base-patch32"
detecteur = pipeline("zero-shot-object-detection", model=MODELE_DETECTION, dtype="auto")

def detecter(image, texte_cible, seuil_alerte=0.05):
    resultats = detecteur(image, candidate_labels=[texte_cible], threshold=0.0)
    if not resultats:
        print(f"⚠️ Aucune détection pour « {texte_cible} ».")
        return None
    meilleure = resultats[0]
    if meilleure["score"] < seuil_alerte:
        print(f"⚠️ Confiance faible ({meilleure['score']:.2f}) pour « {texte_cible} » "
              "— résultat à vérifier.")
    return meilleure

prediction = detecter(image, "a leaf")
print(prediction)


In [ ]:
def dessiner_box(image, detection, couleur="red"):
    img = image.copy()
    draw = ImageDraw.Draw(img)
    box = detection["box"]
    draw.rectangle([box["xmin"], box["ymin"], box["xmax"], box["ymax"]], outline=couleur, width=4)
    draw.text((box["xmin"], max(0, box["ymin"] - 18)), f"{detection['label']} {detection['score']:.2f}", fill=couleur)
    return img

dessiner_box(image, prediction)


## 3. Segmenter l'objet détecté

La segmentation isole les pixels de l'objet. Pour rester simple, on prend la boîte détectée
comme point de départ et on applique `grabCut` (OpenCV).


In [ ]:
def segmenter(image, detection):
    arr = np.array(image)
    mask = np.zeros(arr.shape[:2], np.uint8)
    bgd = np.zeros((1, 65), np.float64)
    fgd = np.zeros((1, 65), np.float64)
    box = detection["box"]
    h, w = arr.shape[:2]
    # Rogner la boîte dans l'image en gardant une marge de fond: grabCut plante
    # (cv2.error) si le rectangle couvre tout le cadre (aucun pixel de fond) — ce qui
    # arrive avec une feuille qui remplit l'image. On garde donc >= 1 px de marge.
    x0 = max(1, int(box["xmin"]))
    y0 = max(1, int(box["ymin"]))
    x1 = min(w - 1, int(box["xmax"]))
    y1 = min(h - 1, int(box["ymax"]))
    rect = (x0, y0, max(1, x1 - x0), max(1, y1 - y0))
    try:
        cv2.grabCut(arr, mask, rect, bgd, fgd, 5, cv2.GC_INIT_WITH_RECT)
    except cv2.error:
        print("⚠️ Segmentation impossible pour cette boîte — image d'origine renvoyée.")
        return image
    masque = np.where((mask == 2) | (mask == 0), 0, 1).astype("uint8")
    retour = arr * masque[:, :, np.newaxis]
    return Image.fromarray(retour)

image_segmente = segmenter(image, prediction)
image_segmente


## 4. Suivre l'objet dans le temps (tracking)

Le tracking consiste à suivre la boîte d'un objet d'une image à la suivante. Pour un notebook
pédagogique, on fabrique une mini-séquence à partir de la même image et on lance un tracker
OpenCV.


In [ ]:
def sequence_depuis_image(image, nb_images=5):
    base = np.array(image)
    frames = []
    for i in range(nb_images):
        canvas = np.full_like(base, 245)
        decalage = 18 * i
        h, w = base.shape[:2]
        y0 = min(20 + decalage, max(0, h - 20))
        x0 = min(20 + decalage, max(0, w - 20))
        x1 = min(x0 + w - 80, w)
        y1 = min(y0 + h - 80, h)
        crop = base[:y1 - y0, :x1 - x0]
        canvas[y0:y0 + crop.shape[0], x0:x0 + crop.shape[1]] = crop
        frames.append(canvas)
    return frames

frames = sequence_depuis_image(image)
box = prediction["box"]
init_box = (
    int(box["xmin"]),
    int(box["ymin"]),
    int(box["xmax"] - box["xmin"]),
    int(box["ymax"] - box["ymin"]),
)

tracker = cv2.TrackerCSRT_create() if hasattr(cv2, "TrackerCSRT_create") else cv2.legacy.TrackerCSRT_create()
# On initialise le tracker sur l'image d'origine (où `init_box` est exacte), puis on le
# laisse suivre l'objet à mesure qu'il se déplace dans la séquence fabriquée.
tracker.init(cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR), init_box)

for i, frame in enumerate(frames, start=1):
    ok, boite = tracker.update(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
    print(f"Image {i}: suivi {ok} | boîte = {tuple(round(v, 1) for v in boite)}")


## 5. Détecter sur tout l'échantillon Kaggle

On applique la détection « a leaf » à chacune des photos chargées à l'étape 1, et on observe
la confiance obtenue selon la catégorie réelle de la photo.


In [ ]:
for img, label in echantillon:
    resultat = detecter(img, "a leaf", seuil_alerte=1.0)  # seuil à 1.0: pas d'avertissement répété
    score = resultat["score"] if resultat else 0.0
    print(f"Vrai: {label:10s} | confiance détection 'a leaf' = {score:.2f}")


---
# 🏋️ Exercices

Ces exercices ne visent pas à *écrire du code*, mais à **observer l'effet** du prompt donné
au détecteur.


### 🏋️ Exercice 1 — Le prompt engineering change la confiance

Comparez la confiance obtenue pour `"leaf"`, `"a leaf"` et `"a green plant leaf"` sur la même
image. La formulation du mot-clé change-t-elle vraiment le résultat?


In [ ]:
# 👉 Votre code ici


### ✅ Solution 1


In [ ]:
for texte in ["leaf", "a leaf", "a green plant leaf"]:
    resultat = detecter(image, texte, seuil_alerte=1.0)
    score = resultat["score"] if resultat else 0.0
    print(f"{texte:22s} → confiance = {score:.3f}")


### 🏋️ Exercice 2 — Comparer plusieurs mots-clés

Comparez les scores de confiance obtenus pour `"a leaf"`, `"a plant"` et `"a fruit"` sur la
même image.


In [ ]:
# 👉 Votre code ici


### ✅ Solution 2


In [ ]:
for mot in ["a leaf", "a plant", "a fruit"]:
    resultat = detecter(image, mot, seuil_alerte=1.0)
    score = resultat["score"] if resultat else 0.0
    print(f"{mot:10s} → confiance = {score:.2f}")


## ✅ Récapitulatif

- La **détection ouverte (OV)** permet de chercher un objet avec un mot, sans réentraînement.
- La **formulation du mot-clé** (prompt engineering) change nettement la confiance obtenue.
- La **segmentation** transforme une boîte en masque de pixels, le **tracking** suit l'objet
  d'image en image.
- Un modèle **petit (~150M paramètres)** suffit pour ce trio, et reste rapide sur Colab.

**➡️ Notebook suivant: `05_LLM_quantization.ipynb`** — faire tourner un grand modèle de
langage (~9 Md paramètres) grâce à la quantification, en local ou via une API cloud (Groq).
